<a href="https://colab.research.google.com/github/Mario-Cattaneo/SemesterProject/blob/main/Splitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import os
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision.transforms import Resize, CenterCrop, ToTensor, Normalize, Compose
from PIL import Image

# 1. Clone the repository if it doesn't exist
REPO_DIR = "imagenet-sample-images"
REPO_URL = "https://github.com/EliSchwartz/imagenet-sample-images.git"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

# =====================================================================
# 2. Define the Edge (Communication Channel) Module
# =====================================================================
class CommunicationEdge(nn.Module):
    """
    Represents a communication link between two devices.
    Can be extended with trainable coders (compression) and noisy channels.
    """
    def __init__(self, source_id: str, target_id: str):
        super().__init__()
        self.source_id = source_id
        self.target_id = target_id
        self.encoder = nn.Identity()  # Trainable compression encoder
        self.channel = nn.Identity()  # Trainable/differentiable noisy channel
        self.decoder = nn.Identity()  # Trainable decompression decoder

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.encoder(x)
        x = self.channel(x)
        x = self.decoder(x)
        return x

# =====================================================================
# 3. Define the Device Node Module
# =====================================================================
class DeviceNode(nn.Module):
    """
    Represents a physical or virtual device in the network.
    """
    def __init__(self, node_id: str, weight: float):
        super().__init__()
        self.node_id = node_id
        self.weight = weight
        self.layers = nn.Sequential()  # Allocated vertical slice of the model
        self.edges = nn.ModuleDict()   # Outgoing edges to successor nodes

    def add_successor(self, successor_node: 'DeviceNode'):
        edge_id = f"{self.node_id}->{successor_node.node_id}"
        self.edges[edge_id] = CommunicationEdge(self.node_id, successor_node.node_id)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Standard sequential forward pass of the allocated layers
        return self.layers(x)

# =====================================================================
# 4. Define the Distributed Graph Coordinator
# =====================================================================
class DistributedGraph(nn.Module):
    def __init__(self, model: nn.Module):
        super().__init__()
        self.nodes = nn.ModuleDict()
        self._allocate_network(model)

    def _allocate_network(self, model: nn.Module):
        """
        Builds the pointer-based graph and allocates VGG-16 layers.
        """
        # Create Nodes
        self.nodes['P'] = DeviceNode(node_id='P', weight=1.0)
        self.nodes['A'] = DeviceNode(node_id='A', weight=0.6)
        self.nodes['B'] = DeviceNode(node_id='B', weight=0.4)
        self.nodes['S'] = DeviceNode(node_id='S', weight=1.0)

        # Establish Pointer-Based Edges (P -> [A, B] -> S)
        self.nodes['P'].add_successor(self.nodes['A'])
        self.nodes['P'].add_successor(self.nodes['B'])
        self.nodes['A'].add_successor(self.nodes['S'])
        self.nodes['B'].add_successor(self.nodes['S'])

        # Allocate Vertical Layers from VGG-16
        # Node P: First Conv block
        self.nodes['P'].layers = model.features[0:5]

        # Nodes A and B: Second Conv block
        self.nodes['A'].layers = model.features[5:10]
        self.nodes['B'].layers = model.features[5:10]

        # Node S: Rest of the features block + classifier
        self.nodes['S'].layers = nn.Sequential(
            model.features[10:],
            model.avgpool,
            nn.Flatten(1),
            model.classifier
        )

    def print_structure(self):
        """Prints the graph topology and layer allocations."""
        print("\n" + "="*80)
        print("DISTRIBUTED GRAPH TOPOLOGY & ALLOCATIONS")
        print("="*80)
        for node_id, node in self.nodes.items():
            print(f"\nDevice Node: '{node_id}' | Horizontal Weight: {node.weight}")
            print(f" -> Allocated Layers:\n{node.layers}")
            if len(node.edges) > 0:
                print(" -> Outgoing Edges:")
                for edge_id, edge in node.edges.items():
                    print(f"    * {edge_id} (Encoder: {type(edge.encoder).__name__} | Channel: {type(edge.channel).__name__})")
        print("="*80 + "\n")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # --- 1. Execute Predecessor Node 'P' ---
        x_P = self.nodes['P'](x)

        # --- 2. Branching & Horizontal Splitting ---
        B, C, H, W = x_P.shape
        weight_A = self.nodes['A'].weight

        h_A = int(round(weight_A * H))
        h_B = H - h_A

        # Standard overlap size (1 row of overlap)
        pad_size = 1

        # Slice with overlap
        slice_A = x_P[:, :, 0 : h_A + pad_size, :]
        slice_B = x_P[:, :, h_A - pad_size : H, :]

        # --- 3. Transmission across Edges (P -> A, P -> B) ---
        edge_PA = self.nodes['P'].edges['P->A']
        edge_PB = self.nodes['P'].edges['P->B']

        transmitted_A = edge_PA(slice_A)
        transmitted_B = edge_PB(slice_B)

        # --- 4. Execute Parallel Nodes 'A' and 'B' ---
        out_A = self.nodes['A'](transmitted_A)
        out_B = self.nodes['B'](transmitted_B)

        # --- 5. Transmission across Edges (A -> S, B -> S) ---
        edge_AS = self.nodes['A'].edges['A->S']
        edge_BS = self.nodes['B'].edges['B->S']

        transmitted_AS = edge_AS(out_A)
        transmitted_BS = edge_BS(out_B)

        # --- 6. Merge & Execute Successor Node 'S' ---
        # Crop out the overlap padding before merging
        # Since MaxPool2d (stride=2) was executed, the height is halved
        clean_A = transmitted_AS[:, :, 0 : h_A // 2, :]
        clean_B = transmitted_BS[:, :, -h_B // 2 : , :]

        merged_features = torch.cat([clean_A, clean_B], dim=2)

        # Execute Node S
        logits = self.nodes['S'](merged_features)
        return logits

# =====================================================================
# 5. Self-Contained ImageNet Loader
# =====================================================================
def load_imagenet_samples(k=20):
    all_files = [f for f in os.listdir(REPO_DIR) if f.endswith(".JPEG")]
    unique_wnids = sorted(list(set(f.split("_")[0] for f in all_files)))
    wnid_to_idx = {wnid: idx for idx, wnid in enumerate(unique_wnids)}

    selected_files = random.sample(all_files, min(k, len(all_files)))

    transform = Compose([
        Resize(224),
        CenterCrop(224),
        ToTensor(),
        Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    images, targets, names = [], [], []
    for filename in selected_files:
        parts = filename.replace(".JPEG", "").split("_")
        wnid = parts[0]
        class_name = " ".join(parts[1:])
        class_idx = wnid_to_idx[wnid]
        img_path = os.path.join(REPO_DIR, filename)
        try:
            img = Image.open(img_path).convert('RGB')
            images.append(transform(img))
            targets.append(class_idx)
            names.append(class_name)
        except Exception as e:
            print(f"Failed to load {filename}: {e}")

    return torch.stack(images), torch.tensor(targets), names

# =====================================================================
# 6. Main Evaluation Pipeline
# =====================================================================
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running evaluation on: {device}\n")

    # Load Pretrained VGG16
    print("Loading pretrained VGG16 model...")
    vgg16 = models.vgg16(weights=models.VGG16_Weights.DEFAULT).to(device).eval()

    # Initialize the Distributed Graph
    dist_graph = DistributedGraph(vgg16).to(device)
    dist_graph.print_structure()

    # Load k random ImageNet samples
    k_samples = 50
    print(f"Loading {k_samples} random ImageNet samples from local repository...")
    images, targets, names = load_imagenet_samples(k=k_samples)
    images, targets = images.to(device), targets.to(device)

    # Run Inference
    print("\nRunning inference...")
    with torch.no_grad():
        out_base = vgg16(images)
        out_dag = dist_graph(images)

    # Calculate Accuracies
    correct_base = 0
    correct_dag = 0

    for i in range(len(names)):
        target_label = targets[i].item()
        pred_base = out_base[i].argmax().item()
        pred_dag = out_dag[i].argmax().item()

        if pred_base == target_label: correct_base += 1
        if pred_dag == target_label: correct_dag += 1

    # Print final accuracies
    total = len(names)
    print("\n" + "="*50)
    print(f"EVALUATION RESULTS ON {total} IMAGENET SAMPLES")
    print("="*50)
    print(f"Baseline Accuracy (Unsplit): {correct_base/total*100:.2f}%")
    print(f"Distributed Graph Accuracy:  {correct_dag/total*100:.2f}%")
    print("="*50)

if __name__ == "__main__":
    main()

Running evaluation on: cpu

Loading pretrained VGG16 model...

DISTRIBUTED GRAPH TOPOLOGY & ALLOCATIONS

Device Node: 'P' | Horizontal Weight: 1.0
 -> Allocated Layers:
Sequential(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU(inplace=True)
  (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (3): ReLU(inplace=True)
  (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
)
 -> Outgoing Edges:
    * P->A (Encoder: Identity | Channel: Identity)
    * P->B (Encoder: Identity | Channel: Identity)

Device Node: 'A' | Horizontal Weight: 0.6
 -> Allocated Layers:
Sequential(
  (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (6): ReLU(inplace=True)
  (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (8): ReLU(inplace=True)
  (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
)
 -> Outgoing Edges:
    * A->S (Encoder: Identity | Ch